# Phase 2: Cleaning and Feature Engineering

## Project context

This notebook parses collected Philippines macro files, inventories additional local Excel sources, standardizes dates and columns, and creates an inflation-first monthly indicator table. It does not train forecasts or build the dashboard.

## Phase 2 objective

- Inspect raw Excel files under `data/raw/`.
- Parse BSP monthly inflation and peso-dollar data where formats are clear.
- Parse annual World Bank context indicators.
- Build a monthly inflation-first table with simple lag, rolling, and change features.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.config import RAW_DATA_DIR, PROCESSED_DATA_DIR, OUTPUTS_DIR, FIGURES_DIR
from src.cleaning import (
    list_excel_files,
    inspect_excel_workbook,
    parse_bsp_inflation,
    parse_bsp_peso_dollar,
    parse_world_bank_csv,
    save_clean_dataset,
)
from src.features import build_inflation_feature_table
from src.visualization import (
    plot_time_series,
    plot_indicator_correlation,
    plot_missingness_summary,
)

INDICATORS_DIR = OUTPUTS_DIR / "indicators"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
INDICATORS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def save_plotly_or_pillow(fig, output_path, chart_type, data):
    output_path = Path(output_path)
    try:
        fig.write_image(str(output_path), width=1200, height=700, scale=2)
        return "plotly"
    except Exception:
        draw_basic_png(output_path, chart_type, data)
        return "pillow_fallback"


def draw_basic_png(output_path, chart_type, data):
    width, height = 1200, 700
    margin = 80
    image = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(image)
    font = ImageFont.load_default()
    draw.text((margin, 24), output_path.stem.replace("_", " ").title(), fill="black", font=font)
    draw.text((margin, 48), "Rendered with Pillow fallback because Plotly static export was unavailable.", fill="#555555", font=font)
    left, top, right, bottom = margin, 110, width - margin, height - margin
    draw.rectangle((left, top, right, bottom), outline="#333333")

    if chart_type == "line":
        frame = data.dropna().copy()
        if len(frame) > 260:
            frame = frame.iloc[np.linspace(0, len(frame) - 1, 260).astype(int)]
        values = frame.iloc[:, 1].astype(float).to_numpy()
        if len(values) > 1:
            vmin, vmax = np.nanmin(values), np.nanmax(values)
            if np.isclose(vmin, vmax):
                vmin, vmax = vmin - 1, vmax + 1
            points = []
            for i, value in enumerate(values):
                x = left + (right - left) * i / max(len(values) - 1, 1)
                y = bottom - (bottom - top) * ((value - vmin) / (vmax - vmin))
                points.append((x, y))
            draw.line(points, fill="#1f77b4", width=2)
    elif chart_type == "bar":
        frame = data.copy()
        values = frame["missing_values"].astype(float).to_numpy()
        labels = frame["column"].astype(str).to_list()
        vmax = max(np.nanmax(values), 1)
        bar_width = (right - left) / max(len(values), 1) * 0.7
        for i, value in enumerate(values):
            x = left + (right - left) * (i + 0.5) / len(values)
            y = bottom - (bottom - top) * (value / vmax)
            draw.rectangle((x - bar_width / 2, y, x + bar_width / 2, bottom), fill="#1f77b4")
            draw.text((x - 30, bottom + 8), labels[i][:10], fill="black", font=font)
    elif chart_type == "heatmap":
        corr = data.copy()
        labels = corr.columns.to_list()
        n = len(labels)
        cell = min((right - left) / max(n, 1), (bottom - top) / max(n, 1))
        for i, row in enumerate(labels):
            for j, col in enumerate(labels):
                value = corr.loc[row, col]
                red = int(255 * max(value, 0))
                blue = int(255 * abs(min(value, 0)))
                green = int(220 * (1 - abs(value)))
                x0 = left + j * cell
                y0 = top + i * cell
                draw.rectangle((x0, y0, x0 + cell, y0 + cell), fill=(red, green, blue), outline="white")
                draw.text((x0 + 4, y0 + 4), f"{value:.2f}", fill="black", font=font)
    image.save(output_path)


def quality_summary(dataset, df, date_col="date", status="created", notes=""):
    start_date = None
    end_date = None
    duplicate_dates = 0
    if date_col in df.columns:
        if date_col == "year":
            years = pd.to_numeric(df[date_col], errors="coerce")
            start_date = int(years.min()) if years.notna().any() else None
            end_date = int(years.max()) if years.notna().any() else None
            duplicate_dates = int(years.duplicated().sum())
        else:
            dates = pd.to_datetime(df[date_col], errors="coerce")
            start_date = dates.min()
            end_date = dates.max()
            duplicate_dates = int(dates.duplicated().sum())
    return {
        "dataset": dataset,
        "rows": len(df),
        "columns": len(df.columns),
        "start_date": start_date,
        "end_date": end_date,
        "missing_values_total": int(df.isna().sum().sum()),
        "duplicate_dates": duplicate_dates,
        "status": status,
        "notes": notes,
    }


## Raw file inventory

In [ ]:
excel_files = list_excel_files(RAW_DATA_DIR)
excel_inventory = pd.DataFrame([inspect_excel_workbook(path) for path in excel_files])
excel_inventory_path = INDICATORS_DIR / "raw_excel_inventory.csv"
excel_inventory.to_csv(excel_inventory_path, index=False)
display(excel_inventory)
print(f"Excel files inspected: {len(excel_inventory)}")

## Additional local Excel file inspection

Additional local Excel files are inventoried. They are not merged into the core monthly table unless their structure is clearly useful and safely parseable for the inflation-first MVP.

In [ ]:
additional_excel_files = [path for path in excel_files if "additional_sources" in str(path)]
for path in additional_excel_files:
    print(path.name)


## Parse BSP monthly inflation

In [ ]:
primary_inflation_path = RAW_DATA_DIR / "bsp_inflation_infrate.xls"
modern_inflation_path = RAW_DATA_DIR / "additional_sources" / "infrate2018.xls"

primary_inflation = parse_bsp_inflation(primary_inflation_path)
inflation_source_used = primary_inflation_path
monthly_inflation = primary_inflation

if modern_inflation_path.exists():
    modern_inflation = parse_bsp_inflation(modern_inflation_path)
    if modern_inflation["date"].max() > primary_inflation["date"].max():
        monthly_inflation = modern_inflation
        inflation_source_used = modern_inflation_path

monthly_inflation_path = PROCESSED_DATA_DIR / "monthly_inflation.csv"
save_clean_dataset(monthly_inflation, monthly_inflation_path)
display(monthly_inflation.head())
display(monthly_inflation.tail())
print(f"Inflation source used: {inflation_source_used}")
print(f"Rows: {len(monthly_inflation):,}; range: {monthly_inflation['date'].min().date()} to {monthly_inflation['date'].max().date()}")

## Parse BSP peso-dollar exchange rate

In [ ]:
monthly_usd_php = parse_bsp_peso_dollar(RAW_DATA_DIR / "bsp_peso_dollar.xlsx")
monthly_usd_php_path = PROCESSED_DATA_DIR / "monthly_usd_php.csv"
save_clean_dataset(monthly_usd_php, monthly_usd_php_path)
display(monthly_usd_php.head())
display(monthly_usd_php.tail())
print(f"Rows: {len(monthly_usd_php):,}; range: {monthly_usd_php['date'].min().date()} to {monthly_usd_php['date'].max().date()}")

## Parse World Bank annual context indicators

In [ ]:
annual_context_parts = [
    parse_world_bank_csv(RAW_DATA_DIR / "world_bank_gdp_growth.csv", "gdp_growth"),
    parse_world_bank_csv(RAW_DATA_DIR / "world_bank_unemployment.csv", "unemployment_rate"),
    parse_world_bank_csv(RAW_DATA_DIR / "world_bank_inflation_backup.csv", "inflation_backup"),
    parse_world_bank_csv(RAW_DATA_DIR / "world_bank_remittances_pct_gdp.csv", "remittances_pct_gdp"),
]

annual_macro_context = annual_context_parts[0]
for part in annual_context_parts[1:]:
    annual_macro_context = annual_macro_context.merge(part, on="year", how="outer")
annual_macro_context = annual_macro_context.sort_values("year").reset_index(drop=True)
annual_macro_context_path = PROCESSED_DATA_DIR / "annual_macro_context.csv"
save_clean_dataset(annual_macro_context, annual_macro_context_path)
display(annual_macro_context.tail())

## Standardize monthly dates

In [ ]:
monthly_inflation["date"] = pd.to_datetime(monthly_inflation["date"])
monthly_usd_php["date"] = pd.to_datetime(monthly_usd_php["date"])
monthly_inflation["date"] = monthly_inflation["date"].dt.to_period("M").dt.to_timestamp()
monthly_usd_php["date"] = monthly_usd_php["date"].dt.to_period("M").dt.to_timestamp()
print(monthly_inflation.dtypes)
print(monthly_usd_php.dtypes)

## Build inflation-first monthly indicator table

In [ ]:
monthly_macro = monthly_inflation.merge(
    monthly_usd_php[["date", "usd_php"]], on="date", how="left"
)
monthly_macro = monthly_macro.sort_values("date").reset_index(drop=True)
display(monthly_macro.head())
display(monthly_macro.tail())

## Add lag, rolling, and change features

In [ ]:
monthly_macro_features = build_inflation_feature_table(monthly_macro)
monthly_macro_path = PROCESSED_DATA_DIR / "monthly_macro_indicators.csv"
save_clean_dataset(monthly_macro_features, monthly_macro_path)
display(monthly_macro_features.head(10))
display(monthly_macro_features.tail())
monthly_macro_features.columns.tolist()

## Data quality review

In [ ]:
quality_rows = [
    quality_summary("monthly_inflation", monthly_inflation, notes=f"Source used: {inflation_source_used.name}"),
    quality_summary("monthly_usd_php", monthly_usd_php),
    quality_summary("annual_macro_context", annual_macro_context, date_col="year", notes="Annual World Bank context indicators."),
    quality_summary("monthly_macro_indicators", monthly_macro_features, notes="Inflation-first monthly feature table."),
]
data_quality_summary = pd.DataFrame(quality_rows)
data_quality_summary_path = INDICATORS_DIR / "data_quality_summary.csv"
data_quality_summary.to_csv(data_quality_summary_path, index=False)
display(data_quality_summary)

## Initial indicator charts

In [ ]:
figure_exports = {}

fig = plot_time_series(monthly_inflation, "date", "inflation_rate", "Philippines Monthly Inflation Rate")
figure_exports["inflation_time_series.png"] = save_plotly_or_pillow(
    fig, FIGURES_DIR / "inflation_time_series.png", "line", monthly_inflation[["date", "inflation_rate"]]
)

fig = plot_time_series(monthly_usd_php, "date", "usd_php", "Monthly USD/PHP Average Exchange Rate")
figure_exports["usd_php_time_series.png"] = save_plotly_or_pillow(
    fig, FIGURES_DIR / "usd_php_time_series.png", "line", monthly_usd_php[["date", "usd_php"]]
)

missing_data = monthly_macro_features.isna().sum().reset_index().rename(columns={"index": "column", 0: "missing_values"})
fig = plot_missingness_summary(monthly_macro_features)
figure_exports["inflation_features_missingness.png"] = save_plotly_or_pillow(
    fig, FIGURES_DIR / "inflation_features_missingness.png", "bar", missing_data
)

numeric_cols = monthly_macro_features.select_dtypes(include="number").columns.tolist()
if len(numeric_cols) >= 2:
    corr = monthly_macro_features[numeric_cols].corr()
    fig = plot_indicator_correlation(monthly_macro_features[numeric_cols])
    figure_exports["macro_indicator_correlation.png"] = save_plotly_or_pillow(
        fig, FIGURES_DIR / "macro_indicator_correlation.png", "heatmap", corr
    )

figure_exports

## Phase 2 limitations

- Forecasting is not performed in this phase.
- Policy rate data remains manual or future-parser work.
- Additional local Excel files were inventoried, but only the clearly parseable BSP inflation and peso-dollar structures were used in the core table.
- World Bank indicators are annual and are kept as context rather than merged into the monthly feature table.

## Next steps for Phase 3 baseline forecasting

- Define the forecasting target, likely one-month-ahead inflation.
- Split data into train/test periods.
- Build simple baseline models using lag and rolling features.
- Compare results against naive inflation benchmarks.